In [5]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [6]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['SebWouters/CheMPS2',
    'f-fathurrahman/PWDFT.jl',
    'nenadmarkus/wlrn',
    'aidenlab/juicer',
    'qiboteam/qibojit',
    'nuest/o2r-project.github.io',
    'rednaga/APKiD',
    'ICRAR/shark',
    'yixuan/spectra',
    'mathLab/PyDMD']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,002818006570601b737a1c290afd875394c116b1,sebwouters_chemps2


In [7]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  1%|▊                                                                               | 10/1052 [00:10<17:56,  1.03s/it]

Got Errors {'1f81f66ceff71908e157caf3288cb7dc6fb209c2': 'Key 1f81f66ceff71908e157caf3288cb7dc6fb209c2 not found in /da5_fast/All.sha1c/commit_31.tch'}


  1%|▉                                                                               | 13/1052 [00:13<17:55,  1.04s/it]

Got Errors {'2791ecba4a82810024577e1fdc8056a77d82f1f1': 'Key 2791ecba4a82810024577e1fdc8056a77d82f1f1 not found in /da5_fast/All.sha1c/commit_39.tch', '27f41de4d0e81427f05e9c38e52d6dfafe1d95ed': 'Key 27f41de4d0e81427f05e9c38e52d6dfafe1d95ed not found in /da5_fast/All.sha1c/commit_39.tch', '2916acbb8ee77acd651ffe4fc14487945ee30c46': 'Key 2916acbb8ee77acd651ffe4fc14487945ee30c46 not found in /da5_fast/All.sha1c/commit_41.tch'}


  2%|█▏                                                                              | 16/1052 [00:16<17:42,  1.03s/it]

...

100%|█████████████████████████████████████████████████████████████████████████████▋| 1047/1052 [18:07<00:05,  1.05s/it]

Got Errors {'f5536a08e0969ed802ab28296cdce5266ff252c4': 'Key f5536a08e0969ed802ab28296cdce5266ff252c4 not found in /da5_fast/All.sha1c/commit_117.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 1052/1052 [18:12<00:00,  1.04s/it]

Got Errors {'ff2fb8467934a43a1d44637ed7491070b8beb6f2': 'Key ff2fb8467934a43a1d44637ed7491070b8beb6f2 not found in /da5_fast/All.sha1c/commit_127.tch'}


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,002818006570601b737a1c290afd875394c116b1,083bf6ba5b716a1a008e60baabb6aa70711739ad,[f259ba4ab5bc62813a4ad788e09459d7e9600b1c],Sebastian Wouters <sebastianwouters@gmail.com>,1464086967,+0200,Sebastian Wouters <sebastianwouters@gmail.com>,1464086967,+0200,Faster Wigner routine - deprecate prime factor...,002818006570601b737a1c290afd875394c116b1,sebwouters_chemps2
1,00348b233ce6cc54dd18625fe994d25b09ed9b71,6da5f0cebe0b51b414298d77490a3d48b7e876b6,[210e4100d3c545e85b62e5257dcc521f45935de3],Sebastian Wouters <sebastianwouters@gmail.com>,1463134405,+0200,Sebastian Wouters <sebastianwouters@gmail.com>,1463134405,+0200,Reshuffle 3-RDM internally\n,00348b233ce6cc54dd18625fe994d25b09ed9b71,sebwouters_chemps2


In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [8]:
print("Total SHA1s requested:", len(df))
print("Total commits successfully retrieved:", len(df_commit_data))
print("Number of projects retrieved:", df_commit_data['project'].nunique())

print("\nCommits successfully retrieved per project:")
print(df_commit_data.groupby('project').size())

Total SHA1s requested: 10516
Total commits successfully retrieved: 10481
Number of projects retrieved: 10

Commits successfully retrieved per project:
project
aidenlab_juicer                 510
f-fathurrahman_pwdft.jl        2112
icrar_shark                    1595
mathlab_pydmd                  1130
nenadmarkus_wlrn                322
nuest_o2r-project.github.io     670
qiboteam_qibojit               1304
rednaga_apkid                  1068
sebwouters_chemps2              771
yixuan_spectra                  999
dtype: int64


In [9]:
summary = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

summary.columns = [
    'project_wocid',
    'commit_sha1',
    'author',
    'time',
    'commit message'
]

summary.to_csv(
    'ddemarco_project_summary.csv',
    sep=';',
    index=False
)

print("Rows saved:", len(summary))
print("Projects saved:", summary['project_wocid'].nunique())

summary.head()

Rows saved: 10481
Projects saved: 10


,project_wocid,commit_sha1,author,time,commit message
0,sebwouters_chemps2,002818006570601b737a1c290afd875394c116b1,Sebastian Wouters <sebastianwouters@gmail.com>,1464086967,Faster Wigner routine - deprecate prime factor...
1,sebwouters_chemps2,00348b233ce6cc54dd18625fe994d25b09ed9b71,Sebastian Wouters <sebastianwouters@gmail.com>,1463134405,Reshuffle 3-RDM internally\n
2,sebwouters_chemps2,00620282a5d6b1e11c3fd3b4f6013e2a9a12fc08,Toon Verstraelen <Toon.Verstraelen@UGent.be>,1430003605,Minor corrections in PyCheMPS2 installation in...
3,sebwouters_chemps2,00687e9aa79b1aa8d6b7770cb9c612378934919f,Sebastian Wouters <sebastianwouters@gmail.com>,1423523915,DMRGSCF plugin for psi4 works :-)\n
4,sebwouters_chemps2,01e5ce2ee76a6bc080c14ef9f48a7f2c4913d3cf,Sebastian Wouters <sebastianwouters@gmail.com>,1464096136,Official version 1.7 release\n


In [10]:
woc_stats = (
    summary.groupby('project_wocid')
    .agg(
        number_of_commits=('commit_sha1', 'nunique'),
        number_of_authors=('author', 'nunique'),
        min_time=('time', 'min'),
        max_time=('time', 'max')
    )
    .reset_index()
)

# Convert Unix timestamps into readable dates
woc_stats['min_time'] = pd.to_datetime(
    woc_stats['min_time'],
    unit='s'
)

woc_stats['max_time'] = pd.to_datetime(
    woc_stats['max_time'],
    unit='s'
)

woc_stats

,project_wocid,number_of_commits,number_of_authors,min_time,max_time
0,aidenlab_juicer,510,44,2015-12-24 18:59:23,2025-09-24 16:22:25
1,f-fathurrahman_pwdft.jl,2112,7,2018-02-19 03:45:59,2025-08-21 11:09:22
2,icrar_shark,1595,30,2017-04-20 04:44:19,2025-07-08 04:15:11
3,mathlab_pydmd,1130,52,2017-06-12 12:33:47,2023-11-16 18:43:11
4,nenadmarkus_wlrn,322,10,2016-02-15 16:50:39,2025-07-20 13:49:52
5,nuest_o2r-project.github.io,670,30,2016-01-19 16:18:32,2025-04-23 11:08:15
6,qiboteam_qibojit,1304,26,2021-04-16 11:23:47,2025-11-06 06:07:04
7,rednaga_apkid,1068,76,2016-06-25 01:07:26,2025-11-02 17:17:03
8,sebwouters_chemps2,771,10,2013-11-18 15:36:41,2025-07-15 18:53:34
9,yixuan_spectra,999,42,2015-09-10 03:54:19,2025-08-26 09:33:36


In [11]:
import requests
import pandas as pd

github_repos = [
    'SebWouters/CheMPS2',
    'f-fathurrahman/PWDFT.jl',
    'nenadmarkus/wlrn',
    'aidenlab/juicer',
    'qiboteam/qibojit',
    'nuest/o2r-project.github.io',
    'rednaga/APKiD',
    'ICRAR/shark',
    'yixuan/spectra',
    'mathLab/PyDMD'
]

github_stats = []

for repo in github_repos:
    info = requests.get(f"https://api.github.com/repos/{repo}").json()
    commits = requests.get(
        f"https://api.github.com/repos/{repo}/commits",
        params={"per_page": 1}
    ).json()

    github_stats.append({
        "repository": repo,
        "stars": info.get("stargazers_count"),
        "forks": info.get("forks_count"),
        "last_commit_date": commits[0]["commit"]["committer"]["date"]
            if isinstance(commits, list) and commits else None
    })

github_stats = pd.DataFrame(github_stats)
github_stats

,repository,stars,forks,last_commit_date
0,SebWouters/CheMPS2,73,38,2025-07-15T18:53:34Z
1,f-fathurrahman/PWDFT.jl,122,23,2026-09-17T09:49:51Z
2,nenadmarkus/wlrn,28,8,2025-07-20T13:49:52Z
3,aidenlab/juicer,490,192,2025-08-31T02:59:38Z
4,qiboteam/qibojit,18,4,2026-09-01T04:47:25Z
5,nuest/o2r-project.github.io,1,2,2025-04-23T11:08:15Z
6,rednaga/APKiD,2578,344,2026-09-02T18:55:13Z
7,ICRAR/shark,33,16,2026-09-09T06:28:09Z
8,yixuan/spectra,866,139,2026-07-04T00:55:59Z
9,mathLab/PyDMD,128,11,2025-02-04T14:31:31Z


## Part 1 Project Summary

### 1. SebWouters/CheMPS2
- GitHub Stars: 73
- GitHub Forks: 38
- Last GitHub Commit: 2025-07-15
- WoC Commits: 771
- WoC Authors: 10
- WoC Min Time: 2013-11-18
- WoC Max Time: 2025-07-15

### 2. f-fathurrahman/PWDFT.jl
- GitHub Stars: 122
- GitHub Forks: 23
- Last GitHub Commit: 2026-09-17
- WoC Commits: 2112
- WoC Authors: 7
- WoC Min Time: 2018-02-19
- WoC Max Time: 2025-08-21

### 3. nenadmarkus/wlrn
- GitHub Stars: 28
- GitHub Forks: 8
- Last GitHub Commit: 2025-07-20
- WoC Commits: 322
- WoC Authors: 10
- WoC Min Time: 2016-02-15
- WoC Max Time: 2025-07-20

### 4. aidenlab/juicer
- GitHub Stars: 490
- GitHub Forks: 192
- Last GitHub Commit: 2025-08-31
- WoC Commits: 510
- WoC Authors: 44
- WoC Min Time: 2015-12-24
- WoC Max Time: 2025-09-24

### 5. qiboteam/qibojit
- GitHub Stars: 18
- GitHub Forks: 4
- Last GitHub Commit: 2026-09-01
- WoC Commits: 1304
- WoC Authors: 26
- WoC Min Time: 2021-04-16
- WoC Max Time: 2025-11-06

### 6. nuest/o2r-project.github.io
- GitHub Stars: 1
- GitHub Forks: 2
- Last GitHub Commit: 2025-04-23
- WoC Commits: 670
- WoC Authors: 30
- WoC Min Time: 2016-01-19
- WoC Max Time: 2025-04-23

### 7. rednaga/APKiD
- GitHub Stars: 2578
- GitHub Forks: 344
- Last GitHub Commit: 2026-09-02
- WoC Commits: 1068
- WoC Authors: 76
- WoC Min Time: 2016-06-25
- WoC Max Time: 2025-11-02

### 8. ICRAR/shark
- GitHub Stars: 33
- GitHub Forks: 16
- Last GitHub Commit: 2026-09-09
- WoC Commits: 1595
- WoC Authors: 30
- WoC Min Time: 2017-04-20
- WoC Max Time: 2025-07-08

### 9. yixuan/spectra
- GitHub Stars: 866
- GitHub Forks: 139
- Last GitHub Commit: 2026-07-04
- WoC Commits: 999
- WoC Authors: 42
- WoC Min Time: 2015-09-10
- WoC Max Time: 2025-08-26

### 10. mathLab/PyDMD
- GitHub Stars: 128
- GitHub Forks: 11
- Last GitHub Commit: 2025-02-04
- WoC Commits: 1130
- WoC Authors: 52
- WoC Min Time: 2017-06-12
- WoC Max Time: 2023-11-16